In [13]:
from funcs import set_env
set_env()

Absolutely. Before training, I would make a **debug notebook/script** that verifies every stage independently:

 1. `data`, `train_pairs`, `test_pairs`
2. record lookup
3. character vocabulary
4. tokenization
5. `Dataset`
6. `DataLoader`
7. CNN embedding shapes
8. A/B embedding similarity
9. field-level comparison vectors
10. final MLP logits/probabilities
11. optional embedding visualization with PCA/t-SNE

 There are also **three things to fix in your current code** before doing this:

 - `fields = data.columns.to_list()` probably includes `record_id` and possibly columns you don't want encoded.
- Your `MLPMatcher.forward()` overwrites `e_A` and `e_B` inside the loop.
- Your evaluation function is still written for the old single-field architecture.

 Let's make the pipeline clean first.

---

 # 1\. Imports

 Add these:

```
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import string
import time
```

---

 # 2\. First inspect your actual data

 Before creating anything:

```
data, train_pairs, test_pairs = train_test_sets()

print("=" * 60)
print("DATA")
print("=" * 60)

print("Shape:", data.shape)
print(data.head())
print("\nColumns:")
print(data.columns.tolist())

print("\nIndex:")
print(data.index.name)

print("\n" + "=" * 60)
print("TRAIN PAIRS")
print("=" * 60)

print("Shape:", train_pairs.shape)
print(train_pairs.head())
print(train_pairs.columns.tolist())

print("\n" + "=" * 60)
print("TEST PAIRS")
print("=" * 60)

print("Shape:", test_pairs.shape)
print(test_pairs.head())
print(test_pairs.columns.tolist())
```

 This is important because your entire pipeline assumes:

```
data.index
    ↓
record_id
```

 and:

```
train_pairs.record_id_1
train_pairs.record_id_2
```

 must exist.

---

 # 3\. Define your actual fields explicitly

 Don't do:

```
fields = data.columns.to_list()
```

 because you probably don't want to encode things like `record_id`.

 Instead:

```
fields = [
    "name",
    "address",
    "phone",
    "email"
]
```

 Obviously change these to your actual columns.

 Then check:

```
missing_fields = [
    field for field in fields
    if field not in data.columns
]

print("Missing fields:", missing_fields)
```

 You want:

```
Missing fields: []
```

 Also inspect missing values:

```
print(data[fields].isna().sum())
```

---

 # 4\. Test the character vocabulary

 Your vocabulary is:

```
PAD_IDX = 0
UNK_IDX = 1

chars = string.ascii_lowercase + string.digits + " .,/-'"

char_to_idx = {
    char: idx + 2
    for idx, char in enumerate(chars)
}

vocab_size = len(char_to_idx) + 2
```

 Let's visualize it:

```
print("Vocabulary size:", vocab_size)

for char, idx in list(char_to_idx.items())[:30]:
    print(repr(char), "→", idx)

print("PAD:", PAD_IDX)
print("UNK:", UNK_IDX)
```

 You should have something like:

```
PAD: 0
UNK: 1
'a' → 2
'b' → 3
...
```

---

 # 5\. Test tokenization manually

 Take a real example from your data:

```
example_text = str(data.iloc[0]["name"])

print("Original:")
print(example_text)

tokens = char_tokenizer(example_text, max_len=50)

print("\nToken IDs:")
print(tokens)

print("\nShape:")
print(tokens.shape)
```

 You should get:

```
Original:
John Smith

Token IDs:
tensor([11, 16, 9, ...])

Shape:
torch.Size([50])
```

---

 # 6\. Visualize the tokenization

 It's useful to actually see:

```
def visualize_tokenization(text, max_len=50):

    tokens = char_tokenizer(text, max_len)

    print(f"Original: {text}")
    print()

    print("Position | Character | Token ID")
    print("-" * 35)

    text_lower = str(text).lower()

    for i, token_id in enumerate(tokens):

        if i < len(text_lower):
            char = text_lower[i]
        else:
            char = "<PAD>"

        print(
            f"{i:8d} | {repr(char):9s} | {token_id.item():8d}"
        )
```

 Then:

```
visualize_tokenization("John Smith")
```

 This is an excellent sanity check.

---

 # 7\. Check the Dataset

 Use the corrected Dataset:

```
class EntityPairDataset(Dataset):

    def __init__(
        self,
        data,
        pairs,
        tokenizer,
        fields,
        max_len=50
    ):
        self.data = data
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.fields = fields
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        row = self.pairs.iloc[idx]

        record_A = self.data.loc[row["record_id_1"]]
        record_B = self.data.loc[row["record_id_2"]]

        features_A = {}
        features_B = {}

        for field in self.fields:

            text_A = str(record_A[field])
            text_B = str(record_B[field])

            features_A[field] = self.tokenizer(
                text_A,
                self.max_len
            )

            features_B[field] = self.tokenizer(
                text_B,
                self.max_len
            )

        label = torch.tensor(
            row["label"],
            dtype=torch.float32
        )

        return features_A, features_B, label
```

 Now:

```
train_dataset = EntityPairDataset(
    data=data,
    pairs=train_pairs,
    tokenizer=char_tokenizer,
    fields=fields,
    max_len=50
)
```

 Test one example:

```
features_A, features_B, label = train_dataset[0]

print("Label:", label)

print("\nFields:")
print(features_A.keys())

for field in fields:

    print(f"\n--- {field} ---")

    print("A:")
    print(features_A[field])
    print("Shape:", features_A[field].shape)

    print("B:")
    print(features_B[field])
    print("Shape:", features_B[field].shape)
```

 You should see:

```
--- name ---
A shape: torch.Size([50])
B shape: torch.Size([50])

--- address ---
A shape: torch.Size([50])
B shape: torch.Size([50])
```

 etc.

---

 # 8\. Visualize an actual pair

 This is something I strongly recommend.

```
def show_pair(dataset, idx, fields):

    row = dataset.pairs.iloc[idx]

    record_A = dataset.data.loc[row["record_id_1"]]
    record_B = dataset.data.loc[row["record_id_2"]]

    print("=" * 70)
    print(f"PAIR {idx}")
    print("=" * 70)

    print("Record A:", row["record_id_1"])
    print("Record B:", row["record_id_2"])
    print("LABEL:", row["label"])

    for field in fields:

        print(f"\n[{field}]")

        print("A:", record_A[field])
        print("B:", record_B[field])
```

 Then:

```
show_pair(train_dataset, 0, fields)
```

 You can also inspect positive and negative examples:

```
positive_idx = train_pairs.index[
    train_pairs["label"] == 1
][0]

negative_idx = train_pairs.index[
    train_pairs["label"] == 0
][0]

show_pair(train_dataset, positive_idx, fields)
show_pair(train_dataset, negative_idx, fields)
```

 This is extremely useful in ER because you can verify that your pair generation actually makes sense.

---

 # 9\. Test the DataLoader

```
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

batch_A, batch_B, labels = next(iter(train_loader))

print("Labels:")
print(labels)

print("\nBatch shapes:")

for field in fields:

    print(
        field,
        "A:", batch_A[field].shape,
        "B:", batch_B[field].shape
    )
```

 Expected:

```
name A: torch.Size([8, 50])
name B: torch.Size([8, 50])

address A: torch.Size([8, 50])
address B: torch.Size([8, 50])

phone A: torch.Size([8, 50])
phone B: torch.Size([8, 50])
```

 If this works, your **data → Dataset → DataLoader** pipeline is good.

---

 # 10\. Test the CNN independently

 Before putting it into the entire model:

```
device = get_device()

cnn = CharCNN(
    vocab_size=vocab_size,
    embed_dim=32,
    num_filters=64,
    output_dim=128
).to(device)
```

 Take one field:

```
name_batch = batch_A["name"].to(device)

embedding = cnn(name_batch)

print("Input:")
print(name_batch.shape)

print("\nEmbedding:")
print(embedding.shape)
```

 Expected:

```
Input:
torch.Size([8, 50])

Embedding:
torch.Size([8, 128])
```

 That's your first major checkpoint.

---

 # 11\. Inspect the CNN intermediate representations

 I'd actually modify `CharCNN` temporarily to return intermediate values.

```
class CharCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=32,
        num_filters=64,
        output_dim=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.conv3 = nn.Conv1d(
            embed_dim,
            num_filters,
            kernel_size=3,
            padding=1
        )

        self.conv5 = nn.Conv1d(
            embed_dim,
            num_filters,
            kernel_size=5,
            padding=2
        )

        self.conv7 = nn.Conv1d(
            embed_dim,
            num_filters,
            kernel_size=7,
            padding=3
        )

        self.fc = nn.Sequential(
            nn.Linear(num_filters * 3, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU()
        )

    def forward(self, x, return_intermediate=False):

        # [B, L] → [B, L, embedding_dim]
        x_embed = self.embedding(x)

        # [B, L, E] → [B, E, L]
        x_embed = x_embed.transpose(1, 2)

        conv3 = torch.relu(
            self.conv3(x_embed)
        )

        conv5 = torch.relu(
            self.conv5(x_embed)
        )

        conv7 = torch.relu(
            self.conv7(x_embed)
        )

        feat3 = conv3.max(dim=2)[0]
        feat5 = conv5.max(dim=2)[0]
        feat7 = conv7.max(dim=2)[0]

        concat_feats = torch.cat(
            [feat3, feat5, feat7],
            dim=1
        )

        out_embedding = self.fc(concat_feats)

        if return_intermediate:

            return {
                "embedding_input": x_embed,
                "conv3": conv3,
                "conv5": conv5,
                "conv7": conv7,
                "feat3": feat3,
                "feat5": feat5,
                "feat7": feat7,
                "embedding": out_embedding
            }

        return out_embedding
```

 Now:

```
result = cnn(
    batch_A["name"].to(device),
    return_intermediate=True
)

for key, value in result.items():

    print(
        f"{key:20s}",
        tuple(value.shape)
    )
```

 You'll get something approximately like:

```
embedding_input      (8, 32, 50)
conv3                (8, 64, 50)
conv5                (8, 64, 50)
conv7                (8, 64, 50)
feat3                (8, 64)
feat5                (8, 64)
feat7                (8, 64)
embedding            (8, 128)
```

 That lets you verify every stage.

---

 # 12\. Visualize the learned/initial embedding

 Even before training, you can visualize the embeddings. They won't mean much yet, but this verifies the pipeline.

```
embeddings = result["embedding"].detach().cpu().numpy()

pca = PCA(n_components=2)

embedding_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(7, 6))

plt.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1]
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Name CNN Embeddings")

plt.show()
```

 After training, this becomes much more interesting.

---

 # 13\. Now fix your MLP

 Your current code has a major problem here:

```
for field in e_A.keys():
      e_A = e_A[field]
      e_B = e_B[field]
```

 After the first iteration, `e_A` is no longer a dictionary.

 Use different variable names:

```
class MLPMatcher(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        num_fields=4,
        hidden_dim=128
    ):
        super().__init__()

        input_dim = embed_dim * 4 * num_fields

        self.classifier = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                hidden_dim,
                hidden_dim // 2
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim // 2,
                1
            )
        )

    def forward(
        self,
        embeddings_A,
        embeddings_B
    ):

        field_comparisons = []

        for field in embeddings_A.keys():

            embedding_A = embeddings_A[field]
            embedding_B = embeddings_B[field]

            diff = torch.abs(
                embedding_A - embedding_B
            )

            prod = embedding_A * embedding_B

            field_vector = torch.cat(
                [
                    embedding_A,
                    embedding_B,
                    diff,
                    prod
                ],
                dim=1
            )

            field_comparisons.append(
                field_vector
            )

        pair_vector = torch.cat(
            field_comparisons,
            dim=1
        )

        logits = self.classifier(pair_vector)

        return logits.squeeze(-1)
```

---

 # 14\. Test the complete EntityMatcher

 Use:

```
class EntityMatcher(nn.Module):

    def __init__(
        self,
        vocab_size,
        fields,
        embedding_dim=128
    ):
        super().__init__()

        self.fields = fields

        self.encoders = nn.ModuleDict({

            field: CharCNN(
                vocab_size=vocab_size,
                embed_dim=32,
                num_filters=64,
                output_dim=embedding_dim
            )

            for field in fields
        })

        self.matcher = MLPMatcher(
            embed_dim=embedding_dim,
            num_fields=len(fields),
            hidden_dim=128
        )

    def forward(
        self,
        features_A,
        features_B,
        return_embeddings=False
    ):

        embeddings_A = {}
        embeddings_B = {}

        for field in self.fields:

            embeddings_A[field] = self.encoders[field](
                features_A[field]
            )

            embeddings_B[field] = self.encoders[field](
                features_B[field]
            )

        logits = self.matcher(
            embeddings_A,
            embeddings_B
        )

        if return_embeddings:

            return (
                logits,
                embeddings_A,
                embeddings_B
            )

        return logits
```

 Now initialize:

```
model = EntityMatcher(
    vocab_size=vocab_size,
    fields=fields,
    embedding_dim=128
).to(device)
```

---

 # 15\. Run one complete batch

```
batch_A = {
    field: batch_A[field].to(device)
    for field in fields
}

batch_B = {
    field: batch_B[field].to(device)
    for field in fields
}

labels = labels.to(device)

with torch.no_grad():

    logits, embeddings_A, embeddings_B = model(
        batch_A,
        batch_B,
        return_embeddings=True
    )

print("Logits:")
print(logits)

print("\nLogits shape:")
print(logits.shape)
```

 You should get:

```
Logits shape:
torch.Size([8])
```

---

 # 16\. Inspect every field's embedding

```
for field in fields:

    print(
        f"{field:15s}",
        "A:",
        tuple(embeddings_A[field].shape),
        "B:",
        tuple(embeddings_B[field].shape)
    )
```

 Expected:

```
name            A: (8, 128) B: (8, 128)
address         A: (8, 128) B: (8, 128)
phone           A: (8, 128) B: (8, 128)
email           A: (8, 128) B: (8, 128)
```

 This is a very important checkpoint.

---

 # 17\. Visualize A vs B embeddings

 You can visualize the embeddings for one field:

```
def visualize_field_embeddings(
    embeddings_A,
    embeddings_B,
    labels,
    field
):

    A = embeddings_A[field].detach().cpu().numpy()
    B = embeddings_B[field].detach().cpu().numpy()

    labels = labels.cpu().numpy()

    all_embeddings = np.vstack([A, B])

    pca = PCA(n_components=2)

    coords = pca.fit_transform(all_embeddings)

    n = len(A)

    A_coords = coords[:n]
    B_coords = coords[n:]

    plt.figure(figsize=(8, 7))

    for label in [0, 1]:

        mask = labels == label

        plt.scatter(
            A_coords[mask, 0],
            A_coords[mask, 1],
            label=f"A, label={label}",
            alpha=0.7
        )

        plt.scatter(
            B_coords[mask, 0],
            B_coords[mask, 1],
            marker="x",
            label=f"B, label={label}",
            alpha=0.7
        )

    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.title(f"{field} Embeddings")

    plt.legend()
    plt.grid(alpha=0.2)

    plt.show()
```

 Run:

```
visualize_field_embeddings(
    embeddings_A,
    embeddings_B,
    labels,
    field="name"
)
```

 Again, **before training this won't tell you much**. After training, it becomes useful.

---

 # 18\. Visualize field similarity

 This is probably even more useful for your ER model.

 Calculate cosine similarity between A and B embeddings:

```
import torch.nn.functional as F
```

 Then:

```
def field_similarities(
    embeddings_A,
    embeddings_B,
    fields
):

    similarities = {}

    for field in fields:

        A = embeddings_A[field]
        B = embeddings_B[field]

        sim = F.cosine_similarity(
            A,
            B,
            dim=1
        )

        similarities[field] = sim.detach().cpu()

    return similarities
```

 Then:

```
similarities = field_similarities(
    embeddings_A,
    embeddings_B,
    fields
)
```

 Inspect:

```
for field in fields:

    print(
        field,
        similarities[field]
    )
```

---

 # 19\. Plot similarity by match/non-match

 This is **really useful for debugging your architecture**.

```
def plot_field_similarities(
    similarities,
    labels,
    fields
):

    labels = labels.cpu().numpy()

    fig, axes = plt.subplots(
        1,
        len(fields),
        figsize=(5 * len(fields), 5)
    )

    if len(fields) == 1:
        axes = [axes]

    for ax, field in zip(axes, fields):

        values = similarities[field].numpy()

        match_values = values[labels == 1]
        nonmatch_values = values[labels == 0]

        ax.hist(
            nonmatch_values,
            bins=20,
            alpha=0.6,
            label="Non-match"
        )

        ax.hist(
            match_values,
            bins=20,
            alpha=0.6,
            label="Match"
        )

        ax.set_title(field)
        ax.set_xlabel("Cosine similarity")
        ax.set_ylabel("Count")

        ax.legend()

    plt.tight_layout()
    plt.show()
```

 Run:

```
plot_field_similarities(
    similarities,
    labels,
    fields
)
```

 This can reveal something very interesting.

 For example, you might discover:

```
name

matches     █████████████████
nonmatches  ███████

address

matches     ███████████████
nonmatches  █████████

phone

matches     ███████████████████
nonmatches  ██
```

 That would tell you that **phone embeddings are much more discriminative than address embeddings**.

---

 # 20\. Test the MLP input dimensions

 You should also explicitly inspect the pair vectors.

 For four fields:

```
128 × 4 = 512
```

 per field.

 Then:

```
512 × 4 = 2048
```

 overall.

 Verify it:

```
with torch.no_grad():

    field_vectors = []

    for field in fields:

        A = embeddings_A[field]
        B = embeddings_B[field]

        diff = torch.abs(A - B)
        prod = A * B

        vector = torch.cat(
            [A, B, diff, prod],
            dim=1
        )

        print(
            field,
            "→",
            vector.shape
        )

        field_vectors.append(vector)

    pair_vector = torch.cat(
        field_vectors,
        dim=1
    )

    print("\nFinal pair vector:")
    print(pair_vector.shape)
```

 Expected:

```
name     → torch.Size([8, 512])
address  → torch.Size([8, 512])
phone    → torch.Size([8, 512])
email    → torch.Size([8, 512])

Final pair vector:
torch.Size([8, 2048])
```

 Then:

```
print(
    "MLP expected input:",
    model.matcher.classifier[0].in_features
)
```

 should say:

```
2048
```

---

 # 21\. Finally test the probability

 Because your MLP returns logits:

```
probabilities = torch.sigmoid(logits)
```

 Then:

```
for i in range(len(labels)):

    print(
        f"Pair {i}: "
        f"label={labels[i].item():.0f}, "
        f"probability={probabilities[i].item():.4f}"
    )
```

 Since the model hasn't been trained yet, something like this is normal:

```
Pair 0: label=1, probability=0.51
Pair 1: label=0, probability=0.48
Pair 2: label=1, probability=0.53
...
```

 **Don't expect meaningful predictions yet.**

---

 # 22\. Then do one training step only

 Before writing a complete training loop, I'd make sure backprop works:

```
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
```

 Then:

```
model.train()

batch_A, batch_B, labels = next(iter(train_loader))

batch_A = {
    field: batch_A[field].to(device)
    for field in fields
}

batch_B = {
    field: batch_B[field].to(device)
    for field in fields
}

labels = labels.to(device)

optimizer.zero_grad()

logits = model(
    batch_A,
    batch_B
)

loss = criterion(
    logits,
    labels
)

print("Loss before backward:", loss.item())

loss.backward()

optimizer.step()

print("One optimization step completed.")
```

 If that works, you've validated the whole computational graph.

---

 # 23\. One very useful debugging check: gradients

 After `loss.backward()`:

```
for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        print(
            f"{name:50s}",
            f"mean_grad={parameter.grad.abs().mean().item():.8f}"
        )
```

 You want to see gradients propagating into:

```
encoders.name.embedding.weight
encoders.name.conv3.weight
encoders.name.conv5.weight
encoders.name.conv7.weight
...
encoders.address...
encoders.phone...
matcher.classifier...
```

 This confirms that:

```
MLP
 ↓
field comparison
 ↓
CNN embeddings
 ↓
CNN convolutions
 ↓
character embedding
```

 is actually connected.

---

 # 24\. Your full debugging sequence

 I would run the code in this exact order:

```
1. train_test_sets()
       ↓
2. Inspect data / train_pairs / test_pairs
       ↓
3. Define fields
       ↓
4. Test character vocabulary
       ↓
5. Test char_tokenizer()
       ↓
6. Inspect one pair
       ↓
7. Test EntityPairDataset
       ↓
8. Test DataLoader
       ↓
9. Test one CharCNN
       ↓
10. Inspect CNN intermediate tensors
       ↓
11. Test all field encoders
       ↓
12. Inspect A/B embeddings
       ↓
13. Calculate field cosine similarities
       ↓
14. Build pair vectors
       ↓
15. Check pair-vector dimensions
       ↓
16. Run complete EntityMatcher
       ↓
17. Check logits/probabilities
       ↓
18. Backprop one batch
       ↓
19. Check gradients
       ↓
20. THEN write the training loop
```

 That approach will save you a lot of debugging time.

 ### One particularly important point

 I would **not visualize embeddings and conclude that the model is learning anything until after training**. Before training, the CNN weights are random. The useful visualizations during development are primarily:

 - raw pair values,
- token IDs,
- tensor shapes,
- A/B embedding cosine similarity,
- field-level similarity distributions,
- gradient magnitudes.

 After training, PCA/t-SNE of the embeddings becomes much more informative.

 Also, your `evaluate_baseline_performance()` function should be rewritten for the dictionary-based multi-field batches; the current version will fail because `batch_a` is now a dictionary, not a tensor.

### Data

In [15]:
from blocking import train_test_sets

data, train_pairs, test_pairs = train_test_sets()

print("-" * 60)
print("DATA")

print("Shape:", data.shape)
print("Columns:")
print(data.columns.tolist())
print("Index:")
print(data.index.name)

print("\n" + "-" * 60)
print("TRAIN PAIRS")
print("Shape:", train_pairs.shape)
print(train_pairs.columns.tolist())

print("\n" + "-" * 60)
print("TEST PAIRS")
print("Shape:", test_pairs.shape)
print(test_pairs.columns.tolist())

Loading Candidate Pairs from file...
Total Candidate Pairs from Union of Rule-based and LSH Blocking: 378655
------------------------------------------------------------
DATA
Shape: (50000, 12)
Columns:
['first_name', 'last_name', 'phone', 'dob', 'city', 'state', 'postal_code', 'street_number', 'street_name', 'apt_number', 'email_user', 'email_domain']
Index:
record_id

------------------------------------------------------------
TRAIN PAIRS
Shape: (302924, 3)
['record_id_1', 'record_id_2', 'label']

------------------------------------------------------------
TEST PAIRS
Shape: (75731, 3)
['record_id_1', 'record_id_2', 'label']


In [27]:
for fields in data.columns:
    print(data[fields].str.len().max())

11
20
10
8
23
2
5
5
26
3
25
22


### Test the character vocabulary

In [18]:
import string

PAD_IDX = 0
UNK_IDX = 1
chars = string.ascii_lowercase + string.digits + " .,/-'"
char_to_idx = { char: idx + 2 for idx, char in enumerate(chars) }
vocab_size = len(char_to_idx) + 2

print("Vocabulary size:", vocab_size)

for char, idx in list(char_to_idx.items())[:30]:
    print(repr(char), "→", idx)

print("PAD:", PAD_IDX)
print("UNK:", UNK_IDX)



Vocabulary size: 44
'a' → 2
'b' → 3
'c' → 4
'd' → 5
'e' → 6
'f' → 7
'g' → 8
'h' → 9
'i' → 10
'j' → 11
'k' → 12
'l' → 13
'm' → 14
'n' → 15
'o' → 16
'p' → 17
'q' → 18
'r' → 19
's' → 20
't' → 21
'u' → 22
'v' → 23
'w' → 24
'x' → 25
'y' → 26
'z' → 27
'0' → 28
'1' → 29
'2' → 30
'3' → 31
PAD: 0
UNK: 1


### Tokenization

In [30]:
import torch

def char_tokenizer(text, max_len=26):
      text = str(text).lower()
      tokens = []
      for char in text:
            tokens.append(char_to_idx.get(char, UNK_IDX))
      tokens = tokens[:max_len] # truncate
      tokens += [PAD_IDX] * (max_len - len(tokens)) # padding

      return torch.tensor(tokens, dtype=torch.long)

example_text = str(data.iloc[0]["first_name"])
print("Original:")
print(example_text)
tokens = char_tokenizer(example_text, max_len=50)
print("\nToken IDs:")
print(tokens)
print("\nShape:")
print(tokens.shape)

Original:
danielle

Token IDs:
tensor([ 5,  2, 15, 10,  6, 13, 13,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

Shape:
torch.Size([50])


### Datasets

In [33]:
from feature_extractor import data

train_data, test_data, train_loader, test_loader = data()

features_A, features_B, label = train_data[0]

print("Label:", label)

print("\nFields:")
print(features_A.keys())

for field in fields:

    print(f"\n--- {field} ---")

    print("A:")
    print(features_A[field])
    print("Shape:", features_A[field].shape)

    print("B:")
    print(features_B[field])
    print("Shape:", features_B[field].shape)

Loading Candidate Pairs from file...
Total Candidate Pairs from Union of Rule-based and LSH Blocking: 378655
Label: tensor(1.)

Fields:
dict_keys(['first_name', 'last_name', 'phone', 'dob', 'city', 'state', 'postal_code', 'street_number', 'street_name', 'apt_number', 'email_user', 'email_domain'])

--- e ---
A:


KeyError: 'e'